In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv
from sklearn.metrics import fbeta_score, average_precision_score

# Assuming 'data' is already loaded from Phase 1
device = torch.device('cpu')
data = data.to(device)

# 1. Define the GCN Architecture
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # First message passing layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        # Second message passing layer to output logits
        x = self.conv2(x, edge_index)
        return x

# 2. Define the GraphSAGE Architecture
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphSAGE, self).__init__()
        # SAGE aggregates neighbor features by concatenation or mean pooling
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# 3. Training Loop Function
def train_and_evaluate(model, data, epochs=200):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    # Handle massive class imbalance by weighting the minority (illicit) class heavier
    weight = torch.tensor([1.0, 10.0]).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=weight)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        # Only compute loss on the labeled training nodes
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        if epoch % 50 == 0:
            print(f'Epoch {epoch:>3} | Loss: {loss.item():.4f}')

    # Evaluation phase
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        # Get probabilities for class 1 (illicit) for PR-AUC
        probs = F.softmax(out, dim=1)[:, 1]
        preds = out.argmax(dim=1)

        # Filter for test set
        y_true = data.y[data.test_mask].cpu().numpy()
        y_pred = preds[data.test_mask].cpu().numpy()
        y_prob = probs[data.test_mask].cpu().numpy()

        f2 = fbeta_score(y_true, y_pred, beta=2)
        pr_auc = average_precision_score(y_true, y_prob)
        
    return f2, pr_auc

# 4. Run Experiments
print("--- Training GCN ---")
gcn_model = GCN(data.num_node_features, 64, 2).to(device)
gcn_f2, gcn_pr = train_and_evaluate(gcn_model, data)
print(f"GCN Final -> F2: {gcn_f2:.4f} | PR-AUC: {gcn_pr:.4f}\n")

print("--- Training GraphSAGE ---")
sage_model = GraphSAGE(data.num_node_features, 64, 2).to(device)
sage_f2, sage_pr = train_and_evaluate(sage_model, data)
print(f"GraphSAGE Final -> F2: {sage_f2:.4f} | PR-AUC: {sage_pr:.4f}")

--- Training GCN ---
Epoch   0 | Loss: 0.8200
Epoch  50 | Loss: 0.2249
Epoch 100 | Loss: 0.1835
Epoch 150 | Loss: 0.1640
GCN Final -> F2: 0.5373 | PR-AUC: 0.5037

--- Training GraphSAGE ---
Epoch   0 | Loss: 0.6736
Epoch  50 | Loss: 0.1205
Epoch 100 | Loss: 0.0818
Epoch 150 | Loss: 0.0724
GraphSAGE Final -> F2: 0.6593 | PR-AUC: 0.6899


In [2]:
from torch_geometric.datasets import EllipticBitcoinDataset

# Reload the dataset into memory
dataset = EllipticBitcoinDataset(root='../data/Elliptic')
data = dataset[0]
print("Data loaded successfully!")

Data loaded successfully!
